# Experiment XI - Cross-Dataset Zero-Shot & Few-Shot Generalization

**Question the professor asked:** take the top trained models and test them, *zero-shot* and *few-shot*,
on a **new unseen dataset**. This notebook does exactly that for the **DR grade (0-4)** task -- the only
label space that transfers cleanly across datasets (the 7-lesion scheme is MMRDR-specific).

### Models under test (top-3: 2 CFP + 1 UWF control)
| tag | model | trained on | role |
|-----|-------|-----------|------|
| `MultiScaleDRNet_CFP` | Exp I dual-branch ResNet-50 | MMRDR-**CFP** | CFP task-CNN |
| `RETFound_CFP` | Exp II RETFound ViT-L/16 | MMRDR-**CFP** | CFP foundation model |
| `UWF_SupCon_control` | Exp VI SupCon ResNet-50+FPN | MMRDR-**UWF** | **negative control** (wrong modality) |

### Unseen external datasets (never seen in any MMRDR training)
- **APTOS 2019** (`aptos2019-blindness-detection`) - 3,662 CFP, ICDR grade 0-4, Indian population.
- **IDRiD** "B. Disease Grading" - 413 train / 103 test CFP, grade 0-4.

### Protocol
- **Zero-shot (k=0):** frozen model, its own grade head, `argmax` on the external **held-out test** split.
- **Few-shot (k>0):** freeze the backbone, extract features once, fit a fresh multinomial logistic head on
  **k images per class** drawn from the external **support pool**, predict the same test split. Averaged
  over `N_EPISODES` random support draws -> report **mean +/- std**.

### Honest caveats (read before trusting a number)
1. Zero-shot uses raw `argmax`; external datasets have a very different grade prior, so **QWK** (ordinal,
   prior-robust) is the headline, not raw accuracy.
2. Few-shot at tiny k is high-variance -> we report mean +/- std over >= `N_EPISODES` episodes, never a
   single draw.
3. IDRiD test n=103 -> wide confidence intervals; treat as a directional second opinion to APTOS.
4. The **UWF control is *expected* to fail zero-shot** on CFP -- that failure is the result (it proves the
   UWF features are modality-specific), not a bug. Its *few-shot* row separately probes whether its frozen
   features are still linearly useful for CFP grade.
5. Linear-probe few-shot != full fine-tuning. It is the standard, low-variance, compute-light transfer
   probe; a full-FT variant is coded but off by default.
6. Lesion task is out of scope by design (external datasets don't share the 7-lesion labels).

In [31]:
# ---- environment (import torch FIRST, before timm) ------------------------------------------------
import os, glob, json, time, warnings, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import pandas as pd
warnings.filterwarnings("ignore")

SEED = 42
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device", DEVICE,
      "| GPUs", torch.cuda.device_count())
if DEVICE.type != "cuda":
    print("\n" + "!" * 68)
    print("!! RUNNING ON CPU -- inference (esp. the RETFound ViT-L) will be painfully slow.")
    print("!! Enable it:  Settings (right panel) -> Accelerator -> GPU T4 x2 -> Run All.")
    print("!" * 68 + "\n")
try:
    import timm
    print("timm", timm.__version__)
except Exception as e:
    print("[warn] timm not importable yet:", str(e)[:120])
try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline
    print("sklearn ready")
except Exception as e:
    raise RuntimeError("scikit-learn is required for the few-shot linear probe: " + str(e))

torch 2.10.0+cu128 | device cuda | GPUs 2
timm 1.0.26
sklearn ready


In [32]:
# ---- run configuration ----------------------------------------------------------------------------
# QUICK_PASS trims the k-grid / episode count for a fast smoke run. Flip to False for the full study.
QUICK_PASS   = False
K_SHOTS      = [5, 10] if QUICK_PASS else [1, 5, 10, 20]
N_EPISODES   = 3 if QUICK_PASS else 5
MAX_TEST     = 600 if QUICK_PASS else None   # subsample external TEST split for a quick smoke (None=all)
FINETUNE     = False                          # full fine-tune few-shot variant (linear probe is primary)
BACKUP       = False                          # push results json/plots to a Kaggle dataset at the end
BACKUP_SLUG  = "krrish1008/transfer-zeroshot-fewshot"
OUT_DIR      = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
os.makedirs(os.path.join(OUT_DIR, "results"), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "plots"), exist_ok=True)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Per-model transfer spec. grade_idx = which forward() output is the 5-way grade logits.
#   MultiScaleDRNet / RETFound -> (grade, lesion)      => grade_idx 0
#   UWF_SupConDRNet            -> (lesion, grade, ...)  => grade_idx 1
MODELS = {
    "MultiScaleDRNet_CFP": dict(kind="msdrnet",  size=512, interp="bilinear", mask="circular",
                                grade_idx=0, feat_dim=512,
                                ckpt=["best_multiscale_resnet.pth"]),
    "RETFound_CFP":        dict(kind="retfound", size=512, interp="bicubic",  mask="none",
                                grade_idx=0, feat_dim=1024,
                                ckpt=["best_retfound_cfp_ft512.pth", "last_retfound_cfp_ft512.pth"]),
    "UWF_SupCon_control":  dict(kind="uwf",      size=256, interp="bilinear", mask="none",
                                grade_idx=1, feat_dim=512,
                                ckpt=["best_model (for 3).pth", "best_model_for_3.pth",
                                      "best_supcon.pth"]),
}
DATASETS = ["aptos", "idrid"]
print("QUICK_PASS", QUICK_PASS, "| K_SHOTS", K_SHOTS, "| N_EPISODES", N_EPISODES,
      "| MAX_TEST", MAX_TEST)

QUICK_PASS False | K_SHOTS [1, 5, 10, 20] | N_EPISODES 5 | MAX_TEST None


In [33]:
# ---- CFP preprocessing (verbatim from findings/custom_architecture_resnet.ipynb) ------------------
def apply_circular_mask(pil_img, tightness=0.96):
    '''Bbox-crop the fundus then apply a centred circular mask (MSDRNet's training preprocessing).'''
    img = np.array(pil_img)
    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)
    gray = np.mean(img, axis=2)
    thresh = gray > 10
    if not np.any(thresh):
        return pil_img
    ys, xs = np.argwhere(thresh)[:, 0], np.argwhere(thresh)[:, 1]
    ymin, ymax, xmin, xmax = ys.min(), ys.max(), xs.min(), xs.max()
    crop = img[ymin:ymax + 1, xmin:xmax + 1]
    h, w, _ = crop.shape
    cy, cx = h // 2, w // 2
    r = int(min(cy, cx) * tightness)
    Y, X = np.ogrid[:h, :w]
    mask = np.sqrt((X - cx) ** 2 + (Y - cy) ** 2) <= r
    clean = np.zeros_like(crop); clean[mask] = crop[mask]
    return Image.fromarray(clean)

_INTERP = {"bilinear": transforms.InterpolationMode.BILINEAR,
           "bicubic":  transforms.InterpolationMode.BICUBIC}

def build_eval_tf(size, interp):
    return transforms.Compose([
        transforms.Resize((size, size), interpolation=_INTERP[interp]),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

class ExternalDataset(Dataset):
    '''Generic external-CFP loader. df has columns 'path' (absolute) and 'grade' (0-4).'''
    def __init__(self, df, size, interp, mask):
        self.paths  = df["path"].to_numpy()
        self.grades = df["grade"].to_numpy().astype(np.int64)
        self.size, self.mask = size, mask
        self.tf = build_eval_tf(size, interp)
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        try:
            img = Image.open(self.paths[i]).convert("RGB")
        except Exception:
            img = Image.new("RGB", (self.size, self.size), (0, 0, 0))
        if self.mask == "circular":
            img = apply_circular_mask(img)
        return self.tf(img), int(self.grades[i]), i

In [34]:
# ---- MODEL 1: MultiScaleDRNet (Exp I CFP) -- verbatim, minus the local ImageNet-file load ---------
class MultiScaleDRNet(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights=None)   # weights come from the fine-tuned checkpoint
        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1, self.layer2 = resnet.layer1, resnet.layer2
        self.layer3, self.layer4 = resnet.layer3, resnet.layer4
        self.early_compression = nn.Sequential(
            nn.Conv2d(256, 512, 1, bias=False), nn.BatchNorm2d(512), nn.AdaptiveAvgPool2d((16, 16)))
        self.deep_compression = nn.Sequential(
            nn.Conv2d(2048, 512, 1, bias=False), nn.BatchNorm2d(512))
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.head_a_lesion = nn.Linear(512, 7)
        self.head_b_grade  = nn.Linear(512, 5)
    def _pooled(self, x):
        x = self.stem(x)
        l1 = self.layer1(x); l2 = self.layer2(l1); l3 = self.layer3(l2); l4 = self.layer4(l3)
        fused = F.relu(self.early_compression(l1) + self.deep_compression(l4))
        return self.global_pool(fused).flatten(1)          # [B, 512]
    def features(self, x):
        return self._pooled(x)
    def forward(self, x):
        p = self._pooled(x)
        return self.head_b_grade(p), self.head_a_lesion(p)  # (grade, lesion)

# ---- MODEL 2: RETFoundMultiTask (Exp II CFP) -- verbatim from create_retfound_v2_nb.py ------------
def build_backbone(cfg):
    return timm.create_model(cfg["arch"], pretrained=False, num_classes=0,
                             img_size=cfg["img_size"], global_pool=cfg["global_pool"],
                             drop_path_rate=cfg.get("drop_path", 0.0))

class RETFoundMultiTask(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone = build_backbone(cfg)
        d = self.backbone.num_features                      # 1024
        self.grade_head  = nn.Linear(d, 5)
        self.lesion_head = nn.Linear(d, 7)
    def features(self, x):
        return self.backbone(x)                             # [B, 1024]
    def forward(self, x):
        f = self.backbone(x)
        return self.grade_head(f), self.lesion_head(f)      # (grade, lesion)

RETFOUND_CFG = dict(arch="vit_large_patch16_224", img_size=512, global_pool="avg", drop_path=0.0)

# ---- MODEL 3: UWF_SupConDRNet (Exp VI UWF, control) -- verbatim from create_verdict_nb.py ---------
class UWF_SupConDRNet(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights=None)
        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1, self.layer2 = resnet.layer1, resnet.layer2
        self.layer3, self.layer4 = resnet.layer3, resnet.layer4
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.proj_early = nn.Sequential(nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, 128))
        self.proj_deep  = nn.Sequential(nn.Linear(2048, 512), nn.ReLU(), nn.Linear(512, 128))
        self.lat_l4 = nn.Conv2d(2048, 512, 1); self.lat_l3 = nn.Conv2d(1024, 512, 1)
        self.lat_l2 = nn.Conv2d(512, 512, 1);  self.lat_l1 = nn.Conv2d(256, 512, 1)
        self.classifier_lesions = nn.Linear(512, 7)
        self.classifier_grade   = nn.Linear(512, 5)
    def _pooled(self, x):
        x = self.stem(x)
        l1 = self.layer1(x); l2 = self.layer2(l1); l3 = self.layer3(l2); l4 = self.layer4(l3)
        p4 = self.lat_l4(l4)
        p3 = self.lat_l3(l3) + F.interpolate(p4, size=l3.shape[-2:], mode="bilinear", align_corners=False)
        p2 = self.lat_l2(l2) + F.interpolate(p3, size=l2.shape[-2:], mode="bilinear", align_corners=False)
        p1 = self.lat_l1(l1) + F.interpolate(p2, size=l1.shape[-2:], mode="bilinear", align_corners=False)
        return self.global_pool(p1).flatten(1)              # [B, 512]
    def features(self, x):
        return self._pooled(x)
    def forward(self, x):
        p = self._pooled(x)
        return self.classifier_lesions(p), self.classifier_grade(p), None, None  # (lesion, grade, ..)
print("model classes ready")

model classes ready


In [35]:
# ---- checkpoint discovery + robust load -----------------------------------------------------------
def find_ckpt(name_candidates):
    '''Recursive glob under /kaggle/input for any of the candidate filenames; local cwd fallback.'''
    roots = ["/kaggle/input", "."]
    for nm in name_candidates:
        for root in roots:
            hits = glob.glob(os.path.join(root, "**", nm), recursive=True)
            if hits:
                return sorted(hits, key=os.path.getmtime, reverse=True)[0]
    return None

def _strip(sd):
    for pre in ("module.", "_orig_mod."):
        if sd and all(k.startswith(pre) for k in sd):
            sd = {k[len(pre):]: v for k, v in sd.items()}
    return sd

def _extract_sd(ck):
    if isinstance(ck, dict):
        for key in ("state_dict", "model_state_dict", "model"):
            if key in ck and isinstance(ck[key], dict):
                return ck[key]
    return ck

def load_into(model, path, critical_prefixes):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    sd = _strip(_extract_sd(ck))
    inc = model.load_state_dict(sd, strict=False)
    miss_crit = [k for k in inc.missing_keys if any(k.startswith(p) for p in critical_prefixes)]
    print(f"    loaded {os.path.basename(path)} | missing {len(inc.missing_keys)} "
          f"unexpected {len(inc.unexpected_keys)}")
    if miss_crit:
        raise RuntimeError(f"checkpoint load FAILED: critical keys missing {miss_crit[:6]}")
    return model

def build_model(tag):
    spec = MODELS[tag]
    path = find_ckpt(spec["ckpt"])
    if path is None:
        raise FileNotFoundError(
            f"\n[MISSING CHECKPOINT for '{tag}'] looked for {spec['ckpt']} under /kaggle/input.\n"
            f"  Attach the dataset holding it via 'Add Data'. Expected sources:\n"
            f"   - MultiScaleDRNet_CFP : local findings/best_multiscale_resnet.pth (upload as a dataset)\n"
            f"   - RETFound_CFP        : Kaggle dataset krrish1008/checkpoint-retfound-512\n"
            f"   - UWF_SupCon_control  : local contrastive_learning_lesions_3/'best_model (for 3).pth'\n")
    kind = spec["kind"]
    if kind == "msdrnet":
        model = load_into(MultiScaleDRNet(), path, ["stem.", "layer4.", "head_b_grade."])
    elif kind == "retfound":
        model = load_into(RETFoundMultiTask(RETFOUND_CFG), path, ["backbone.blocks.", "grade_head."])
    elif kind == "uwf":
        model = load_into(UWF_SupConDRNet(), path, ["stem.", "layer4.", "classifier_grade."])
    else:
        raise ValueError(kind)
    return model.to(DEVICE).eval()
print("loaders ready")

loaders ready


In [36]:
# ---- external dataset finders (robust to Kaggle mirror layout differences) ------------------------
def _find_csv(col_substrings):
    '''Return every CSV under /kaggle/input whose columns contain all given substrings (lowercased).'''
    out = []
    for c in glob.glob("/kaggle/input/**/*.csv", recursive=True):
        try:
            df = pd.read_csv(c, nrows=5)
        except Exception:
            continue
        cols = [str(x).lower().strip() for x in df.columns]
        if all(any(sub in col for col in cols) for sub in col_substrings):
            out.append(c)
    return out

def load_aptos():
    '''APTOS 2019. Prefer the clean competition CSV (id_code,diagnosis); else fall back to a
    folder-labelled mirror where subdirs 0..4 under an 'aptos' path encode the grade.'''
    # --- preferred: clean competition dataset (train.csv with id_code,diagnosis) ---
    csvs = _find_csv(["id_code", "diagnosis"])
    if csvs:
        csv = sorted(csvs, key=lambda p: ("train" not in p.lower(), len(p)))[0]
        df = pd.read_csv(csv)
        base = os.path.dirname(csv)
        idx = {}
        for ext in ("png", "jpg", "jpeg"):
            for p in glob.glob(os.path.join(base, "**", "*." + ext), recursive=True):
                idx.setdefault(os.path.splitext(os.path.basename(p))[0], p)
        df["path"] = df["id_code"].astype(str).map(idx.get)
        df = df.dropna(subset=["path"]).rename(columns={"diagnosis": "grade"})
        df["grade"] = df["grade"].astype(int)
        if len(df) >= 100:      # else the CSV is present but its images aren't -> use folder mode
            return df[["path", "grade"]].reset_index(drop=True), os.path.basename(csv) + " (clean)"
    # --- fallback: folder-labelled mirror; find a dir directly holding class folders 0..4 ---
    root = None
    for d in glob.glob("/kaggle/input/**/", recursive=True):
        if "aptos" not in d.lower():
            continue
        subs = {x for x in os.listdir(d) if os.path.isdir(os.path.join(d, x))}
        if {"0", "1", "2", "3", "4"} <= subs:
            root = d
            break
    if root is None:
        raise FileNotFoundError("APTOS not found: neither an id_code/diagnosis CSV nor class folders "
                                "0..4 under an 'aptos' path. Attach 'aptos2019-blindness-detection'.")
    rows = []
    for g in range(5):
        for ext in ("png", "jpg", "jpeg"):
            for p in glob.glob(os.path.join(root, str(g), "**", "*." + ext), recursive=True):
                rows.append((p, g))
    df = pd.DataFrame(rows, columns=["path", "grade"])
    return df.reset_index(drop=True), "APTOS folder-labelled (augmented)"

def _idrid_read(csv):
    '''Read one IDRiD split CSV; resolve images WITHIN that CSV's own folder (Train/Test have
    colliding basenames like IDRiD_0001.jpg, so a global index would cross them).'''
    df = pd.read_csv(csv)
    ren = {}
    for c in df.columns:
        cl = str(c).lower().strip()
        if "image name" in cl or cl == "image":
            ren[c] = "name"
        elif "retinopathy grade" in cl:
            ren[c] = "grade"
    df = df.rename(columns=ren)[["name", "grade"]].dropna()
    folder = os.path.dirname(csv)
    idx = {}
    for ext in ("jpg", "jpeg", "png"):
        for p in glob.glob(os.path.join(folder, "**", "*." + ext), recursive=True):
            b = os.path.basename(p)
            idx.setdefault(b, p)                                # 'Image name' includes the extension
            idx.setdefault(os.path.splitext(b)[0], p)
    def _resolve(n):
        n = str(n).strip()
        return idx.get(n) or idx.get(os.path.splitext(n)[0])
    df["path"] = df["name"].map(_resolve)
    df = df.dropna(subset=["path"])
    df["grade"] = df["grade"].astype(int)
    return df[["path", "grade"]].reset_index(drop=True)

def load_idrid():
    '''IDRiD Disease Grading -> (support_df, test_df). Uses the official Train/Test split folders.'''
    csvs = _find_csv(["retinopathy grade"])
    if not csvs:
        raise FileNotFoundError("IDRiD grading CSV (column 'Retinopathy grade') not found -- attach "
                                "an IDRiD Disease-Grading dataset.")
    which = lambda c: os.path.basename(os.path.dirname(c)).lower()
    train_csv = [c for c in csvs if which(c) == "train"]
    test_csv  = [c for c in csvs if which(c) == "test"]
    if train_csv and test_csv:
        return _idrid_read(train_csv[0]), _idrid_read(test_csv[0]), csvs
    return _idrid_read(csvs[0]), None, csvs         # single CSV -> get_splits does a stratified half

def stratified_half(df, seed=SEED):
    '''Split a labelled df into (support_pool, test) 50/50, stratified on grade.'''
    from sklearn.model_selection import train_test_split
    strat = df["grade"] if df["grade"].value_counts().min() >= 2 else None
    a, b = train_test_split(df, test_size=0.5, random_state=seed, stratify=strat)
    return a.reset_index(drop=True), b.reset_index(drop=True)

def get_splits(dataset):
    '''Return (support_pool_df, test_df) for the given external dataset.'''
    if dataset == "aptos":
        df, csv = load_aptos()
        sup, test = stratified_half(df)
        src = os.path.basename(csv)
    elif dataset == "idrid":
        df_train, df_test, _ = load_idrid()
        if df_test is not None and len(df_test) > 0:
            sup, test = df_train, df_test          # official train=support, test=held-out
            src = "IDRiD grading (official Train/Test)"
        else:
            sup, test = stratified_half(df_train)  # no official test split found
            src = "IDRiD grading (stratified half)"
    else:
        raise ValueError(dataset)
    if MAX_TEST is not None and len(test) > MAX_TEST:
        test = test.sample(n=MAX_TEST, random_state=SEED).reset_index(drop=True)
    hist = lambda d: {int(g): int((d["grade"] == g).sum()) for g in range(5)}
    print(f"[{dataset}] src={src} | support {len(sup)} {hist(sup)} | test {len(test)} {hist(test)}")
    # augmented mirrors can place near-duplicate images across support & test -> few-shot may be
    # OPTIMISTIC (zero-shot, which never fits on the support pool, is unaffected).
    aug = ("augmented" in src.lower()) or (dataset == "idrid" and len(sup) + len(test) > 700)
    if aug:
        print(f"  [caveat] '{dataset}' looks AUGMENTED -> treat FEW-SHOT numbers as optimistic "
              f"(support/test overlap); ZERO-SHOT is unaffected.")
    assert len(sup) > 0 and len(test) > 0, f"empty split for {dataset}"
    return sup, test
print("external finders ready")

external finders ready


In [37]:
# ---- grade metrics (VERBATIM from experiment_07/evaluation/metrics.py) -----------------------------
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, cohen_kappa_score,
                             confusion_matrix, f1_score, matthews_corrcoef, recall_score)

def grade_metrics(preds, targets):
    gp, gt = np.asarray(preds), np.asarray(targets)
    return {
        "accuracy": float(accuracy_score(gt, gp)),
        "balanced_acc": float(balanced_accuracy_score(gt, gp)),
        "quadratic_kappa": float(cohen_kappa_score(gt, gp, weights="quadratic")),
        "macro_f1": float(f1_score(gt, gp, labels=list(range(5)), average="macro", zero_division=0)),
        "mcc": float(matthews_corrcoef(gt, gp)) if len(np.unique(gt)) > 1 else 0.0,
        "per_class_recall": recall_score(gt, gp, labels=list(range(5)), average=None,
                                         zero_division=0).tolist(),
        "confusion_matrix": confusion_matrix(gt, gp, labels=list(range(5))).tolist(),
        "n": int(len(gt)),
    }
print("metrics ready")

metrics ready


In [38]:
# ---- single forward pass -> grade preds (zero-shot) AND penultimate features (few-shot) -----------
from tqdm.auto import tqdm

@torch.no_grad()
def infer_all(model, loader, grade_idx, desc=""):
    all_pred, all_true, all_feat = [], [], []
    for imgs, grades, _ in tqdm(loader, desc=desc, leave=False, dynamic_ncols=True):
        imgs = imgs.to(DEVICE, non_blocking=True)
        out = model(imgs)
        logits = out[grade_idx]
        feats = model.features(imgs)
        all_pred.append(logits.argmax(1).cpu().numpy())
        all_feat.append(feats.float().cpu().numpy())
        all_true.append(grades.numpy())
    return (np.concatenate(all_pred), np.concatenate(all_true),
            np.concatenate(all_feat).astype(np.float32))

def make_loader(df, spec, bs=None):
    # the 3.6GB RETFound ViT-L needs a small batch at 512px to fit a single T4; CNNs can go wider.
    if bs is None:
        bs = 8 if spec["kind"] == "retfound" else 32
    ds = ExternalDataset(df, spec["size"], spec["interp"], spec["mask"])
    return DataLoader(ds, batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
print("inference ready")

inference ready


In [39]:
# ---- few-shot linear-probe episode sampler + fitter -----------------------------------------------
def sample_k_per_class(labels, k, rng):
    '''Return indices with exactly k samples per grade class (or all available if a class has < k).'''
    idx = []
    for c in range(5):
        pool = np.where(labels == c)[0]
        if len(pool) == 0:
            continue
        take = min(k, len(pool))
        idx.extend(rng.choice(pool, size=take, replace=False).tolist())
    return np.array(idx, dtype=int)

def few_shot_eval(feat_sup, y_sup, feat_test, y_test, k, n_episodes, seed=SEED):
    '''Fit a fresh (StandardScaler -> multinomial LogisticRegression) head on k/class support features,
    predict the frozen test features. Averaged over n_episodes support draws.'''
    qwks, accs = [], []
    for ep in range(n_episodes):
        rng = np.random.RandomState(seed * 1000 + ep)
        sidx = sample_k_per_class(y_sup, k, rng)
        if len(np.unique(y_sup[sidx])) < 2:      # need >= 2 classes to fit
            continue
        clf = make_pipeline(StandardScaler(),
                            LogisticRegression(max_iter=2000, C=1.0, class_weight="balanced"))
        clf.fit(feat_sup[sidx], y_sup[sidx])
        pred = clf.predict(feat_test)
        m = grade_metrics(pred, y_test)
        qwks.append(m["quadratic_kappa"]); accs.append(m["accuracy"])
    if not qwks:
        return dict(k=k, qwk_mean=float("nan"), qwk_std=float("nan"),
                    acc_mean=float("nan"), acc_std=float("nan"), episodes=0)
    return dict(k=k, qwk_mean=float(np.mean(qwks)), qwk_std=float(np.std(qwks)),
                acc_mean=float(np.mean(accs)), acc_std=float(np.std(accs)), episodes=len(qwks))
print("few-shot ready")

few-shot ready


In [40]:
# ---- run everything: zero-shot + few-shot for every (dataset x model) -----------------------------
zeroshot, fewshot = {}, {}
for dataset in DATASETS:
    print("\n" + "=" * 70 + f"\nDATASET: {dataset}\n" + "=" * 70)
    try:
        df_sup, df_test = get_splits(dataset)
    except Exception as e:
        print(f"[skip {dataset}] {e}")
        continue
    zeroshot[dataset], fewshot[dataset] = {}, {}
    for tag, spec in MODELS.items():
        print(f"\n--- model {tag} on {dataset} ---")
        try:
            model = build_model(tag)
        except Exception as e:
            print(f"[skip {tag}] {e}")
            continue
        t0 = time.time()
        print(f"  extracting features (test {len(df_test)} + support {len(df_sup)} imgs @ "
              f"{spec['size']}px, bs {'8' if spec['kind']=='retfound' else '32'})...")
        te_pred, te_true, te_feat = infer_all(model, make_loader(df_test, spec), spec["grade_idx"],
                                              desc=f"{tag}/{dataset} test")
        _,        su_true, su_feat = infer_all(model, make_loader(df_sup,  spec), spec["grade_idx"],
                                              desc=f"{tag}/{dataset} support")
        zm = grade_metrics(te_pred, te_true)
        zeroshot[dataset][tag] = zm
        print(f"  ZERO-SHOT  QWK {zm['quadratic_kappa']:.3f} | acc {zm['accuracy']:.3f} | "
              f"bal-acc {zm['balanced_acc']:.3f}  ({time.time()-t0:.0f}s)")
        fewshot[dataset][tag] = []
        for k in K_SHOTS:
            r = few_shot_eval(su_feat, su_true, te_feat, te_true, k, N_EPISODES)
            fewshot[dataset][tag].append(r)
            print(f"  FEW-SHOT k={k:<3d} QWK {r['qwk_mean']:.3f}+/-{r['qwk_std']:.3f} | "
                  f"acc {r['acc_mean']:.3f}+/-{r['acc_std']:.3f} (n_ep {r['episodes']})")
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

with open(os.path.join(OUT_DIR, "results", "zeroshot.json"), "w") as f:
    json.dump(zeroshot, f, indent=2)
with open(os.path.join(OUT_DIR, "results", "fewshot.json"), "w") as f:
    json.dump(fewshot, f, indent=2)
print("\nsaved results/zeroshot.json + results/fewshot.json")


DATASET: aptos
[aptos] src=train.csv (clean) | support 1831 {0: 902, 1: 185, 2: 500, 3: 96, 4: 148} | test 1831 {0: 903, 1: 185, 2: 499, 3: 97, 4: 147}

--- model MultiScaleDRNet_CFP on aptos ---
    loaded best_multiscale_resnet.pth | missing 0 unexpected 0
  extracting features (test 1831 + support 1831 imgs @ 512px, bs 32)...


MultiScaleDRNet_CFP/aptos test:   0%|          | 0/58 [00:00<?, ?it/s]

MultiScaleDRNet_CFP/aptos support:   0%|          | 0/58 [00:00<?, ?it/s]

  ZERO-SHOT  QWK 0.854 | acc 0.705 | bal-acc 0.528  (894s)
  FEW-SHOT k=1   QWK 0.691+/-0.033 | acc 0.589+/-0.035 (n_ep 5)
  FEW-SHOT k=5   QWK 0.749+/-0.038 | acc 0.647+/-0.040 (n_ep 5)
  FEW-SHOT k=10  QWK 0.770+/-0.010 | acc 0.687+/-0.014 (n_ep 5)
  FEW-SHOT k=20  QWK 0.776+/-0.034 | acc 0.705+/-0.013 (n_ep 5)

--- model RETFound_CFP on aptos ---
    loaded best_retfound_cfp_ft512.pth | missing 0 unexpected 0
  extracting features (test 1831 + support 1831 imgs @ 512px, bs 8)...


RETFound_CFP/aptos test:   0%|          | 0/229 [00:00<?, ?it/s]

...

RETFound_CFP/idrid support:   0%|          | 0/155 [00:00<?, ?it/s]

  ZERO-SHOT  QWK 0.593 | acc 0.476 | bal-acc 0.398  (686s)
  FEW-SHOT k=1   QWK 0.589+/-0.074 | acc 0.307+/-0.050 (n_ep 5)
  FEW-SHOT k=5   QWK 0.660+/-0.052 | acc 0.352+/-0.061 (n_ep 5)
  FEW-SHOT k=10  QWK 0.697+/-0.028 | acc 0.432+/-0.048 (n_ep 5)
  FEW-SHOT k=20  QWK 0.711+/-0.034 | acc 0.469+/-0.046 (n_ep 5)

--- model UWF_SupCon_control on idrid ---
    loaded best_model (for 3).pth | missing 0 unexpected 0
  extracting features (test 309 + support 1239 imgs @ 256px, bs 32)...


UWF_SupCon_control/idrid test:   0%|          | 0/10 [00:00<?, ?it/s]

UWF_SupCon_control/idrid support:   0%|          | 0/39 [00:00<?, ?it/s]

  ZERO-SHOT  QWK 0.598 | acc 0.375 | bal-acc 0.431  (128s)
  FEW-SHOT k=1   QWK 0.254+/-0.122 | acc 0.231+/-0.028 (n_ep 5)
  FEW-SHOT k=5   QWK 0.480+/-0.037 | acc 0.296+/-0.025 (n_ep 5)
  FEW-SHOT k=10  QWK 0.552+/-0.028 | acc 0.349+/-0.052 (n_ep 5)
  FEW-SHOT k=20  QWK 0.575+/-0.021 | acc 0.394+/-0.055 (n_ep 5)

saved results/zeroshot.json + results/fewshot.json


In [41]:
# ---- zero-shot summary table ----------------------------------------------------------------------
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print("\nZERO-SHOT (grade 0-4) -- QWK / accuracy on the external held-out test set")
print(f"{'dataset':8s} {'model':22s} {'QWK':>7s} {'acc':>7s} {'bal_acc':>8s} {'macroF1':>8s} {'n':>6s}")
for ds in zeroshot:
    for tag, m in zeroshot[ds].items():
        print(f"{ds:8s} {tag:22s} {m['quadratic_kappa']:7.3f} {m['accuracy']:7.3f} "
              f"{m['balanced_acc']:8.3f} {m['macro_f1']:8.3f} {m['n']:6d}")

# ---- few-shot curves (QWK vs k, zero-shot anchored at k=0) -----------------------------------------
for ds in fewshot:
    plt.figure(figsize=(6, 4))
    for tag in fewshot[ds]:
        ks = [0] + [r["k"] for r in fewshot[ds][tag]]
        ys = [zeroshot[ds][tag]["quadratic_kappa"]] + [r["qwk_mean"] for r in fewshot[ds][tag]]
        es = [0.0] + [r["qwk_std"] for r in fewshot[ds][tag]]
        plt.errorbar(ks, ys, yerr=es, marker="o", capsize=3, label=tag)
    plt.title(f"{ds.upper()} - grade QWK vs shots/class")
    plt.xlabel("k (labeled images per class; k=0 = zero-shot)")
    plt.ylabel("quadratic-weighted kappa")
    plt.grid(alpha=0.3); plt.legend(fontsize=8); plt.tight_layout()
    p = os.path.join(OUT_DIR, "plots", f"fewshot_{ds}.png")
    plt.savefig(p, dpi=130); plt.close()
    print("saved", p)

# ---- zero-shot confusion matrices -----------------------------------------------------------------
for ds in zeroshot:
    tags = list(zeroshot[ds].keys())
    if not tags:
        continue
    fig, axes = plt.subplots(1, len(tags), figsize=(4 * len(tags), 3.6))
    if len(tags) == 1:
        axes = [axes]
    for ax, tag in zip(axes, tags):
        cm = np.array(zeroshot[ds][tag]["confusion_matrix"], dtype=float)
        cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
        ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
        ax.set_title(f"{tag}\nQWK {zeroshot[ds][tag]['quadratic_kappa']:.2f}", fontsize=8)
        ax.set_xlabel("pred"); ax.set_ylabel("true")
        ax.set_xticks(range(5)); ax.set_yticks(range(5))
        for i in range(5):
            for j in range(5):
                ax.text(j, i, f"{cmn[i,j]:.2f}", ha="center", va="center", fontsize=6,
                        color="white" if cmn[i, j] > 0.5 else "black")
    fig.suptitle(f"{ds.upper()} zero-shot confusion (row-normalised)", fontsize=10)
    plt.tight_layout()
    p = os.path.join(OUT_DIR, "plots", f"confusion_{ds}.png")
    plt.savefig(p, dpi=130); plt.close()
    print("saved", p)


ZERO-SHOT (grade 0-4) -- QWK / accuracy on the external held-out test set
dataset  model                      QWK     acc  bal_acc  macroF1      n
aptos    MultiScaleDRNet_CFP      0.854   0.705    0.528    0.505   1831
aptos    RETFound_CFP             0.860   0.711    0.528    0.495   1831
aptos    UWF_SupCon_control       0.689   0.578    0.482    0.412   1831
idrid    MultiScaleDRNet_CFP      0.626   0.511    0.481    0.478    309
idrid    RETFound_CFP             0.593   0.476    0.398    0.404    309
idrid    UWF_SupCon_control       0.598   0.375    0.431    0.364    309
saved /kaggle/working/plots/fewshot_aptos.png
saved /kaggle/working/plots/fewshot_idrid.png
saved /kaggle/working/plots/confusion_aptos.png
saved /kaggle/working/plots/confusion_idrid.png


In [42]:
# ---- printed verdict ------------------------------------------------------------------------------
print("\n" + "=" * 70 + "\nVERDICT - Cross-Dataset Generalization\n" + "=" * 70)
for ds in zeroshot:
    if not zeroshot[ds]:
        continue
    best = max(zeroshot[ds], key=lambda t: zeroshot[ds][t]["quadratic_kappa"])
    print(f"\n[{ds.upper()}]")
    print(f"  Best ZERO-SHOT generalizer: {best} "
          f"(QWK {zeroshot[ds][best]['quadratic_kappa']:.3f})")
    for tag in zeroshot[ds]:
        z = zeroshot[ds][tag]["quadratic_kappa"]
        fs = fewshot[ds].get(tag, [])
        kmax = fs[-1] if fs else None
        if kmax and not np.isnan(kmax["qwk_mean"]):
            delta = kmax["qwk_mean"] - z
            print(f"    {tag:22s} zero-shot QWK {z:6.3f} -> {kmax['k']}-shot "
                  f"{kmax['qwk_mean']:.3f} (delta {delta:+.3f})")
        else:
            print(f"    {tag:22s} zero-shot QWK {z:6.3f}")
    if "UWF_SupCon_control" in zeroshot[ds]:
        uz = zeroshot[ds]["UWF_SupCon_control"]["quadratic_kappa"]
        tag_note = "as expected (modality-specific features)" if uz < 0.4 else "surprisingly transferable"
        print(f"  Control check: UWF model zero-shot QWK {uz:.3f} -> {tag_note}")
print("\nInterpretation: zero-shot = raw cross-domain transfer of each model's own head; "
      "few-shot delta = how much a tiny labelled support set recovers via a linear probe on frozen "
      "features. QWK is the ordinal, prior-robust headline.")

# ---- optional backup ------------------------------------------------------------------------------
if BACKUP:
    try:
        import subprocess, shutil
        stage = os.path.join(OUT_DIR, "transfer_backup"); os.makedirs(stage, exist_ok=True)
        for sub in ("results", "plots"):
            for p in glob.glob(os.path.join(OUT_DIR, sub, "*")):
                shutil.copy(p, os.path.join(stage, os.path.basename(p)))
        json.dump({"title": BACKUP_SLUG.split("/")[-1], "id": BACKUP_SLUG,
                   "licenses": [{"name": "CC0-1.0"}]},
                  open(os.path.join(stage, "dataset-metadata.json"), "w"))
        r = subprocess.run(["kaggle", "datasets", "version", "-p", stage, "-m", "transfer results",
                            "--dir-mode", "zip"], capture_output=True, text=True)
        if r.returncode != 0:
            subprocess.run(["kaggle", "datasets", "create", "-p", stage, "--dir-mode", "zip"],
                           capture_output=True, text=True)
        print("[backup] pushed to", BACKUP_SLUG)
    except Exception as e:
        print("[backup] skipped:", str(e)[:120])


VERDICT - Cross-Dataset Generalization

[APTOS]
  Best ZERO-SHOT generalizer: RETFound_CFP (QWK 0.860)
    MultiScaleDRNet_CFP    zero-shot QWK  0.854 -> 20-shot 0.776 (delta -0.078)
    RETFound_CFP           zero-shot QWK  0.860 -> 20-shot 0.850 (delta -0.009)
    UWF_SupCon_control     zero-shot QWK  0.689 -> 20-shot 0.759 (delta +0.070)
  Control check: UWF model zero-shot QWK 0.689 -> surprisingly transferable

[IDRID]
  Best ZERO-SHOT generalizer: MultiScaleDRNet_CFP (QWK 0.626)
    MultiScaleDRNet_CFP    zero-shot QWK  0.626 -> 20-shot 0.676 (delta +0.050)
    RETFound_CFP           zero-shot QWK  0.593 -> 20-shot 0.711 (delta +0.118)
    UWF_SupCon_control     zero-shot QWK  0.598 -> 20-shot 0.575 (delta -0.023)
  Control check: UWF model zero-shot QWK 0.598 -> surprisingly transferable

Interpretation: zero-shot = raw cross-domain transfer of each model's own head; few-shot delta = how much a tiny labelled support set recovers via a linear probe on frozen features. QWK is the

## How to read these results

**Zero-shot (k=0).** This is the honest generalization number: a model trained only on MMRDR, applied
unchanged to a different hospital's camera and population. A CFP model that keeps a high **QWK** here has
learned transferable DR structure, not dataset-specific shortcuts. Expect some drop vs the in-domain
number -- that gap *is* the domain shift.

**Few-shot (k>0).** Freezing the backbone and fitting a fresh logistic head on a handful of labelled
external images tells you how quickly the domain gap closes and how good the *features* are, independent
of the original head's calibration. A steep rise from k=0 -> k=20 means the representation was fine and
only the head needed re-aiming.

**The UWF control.** It was trained on ultra-widefield images and is applied to standard fundus photos.
Its zero-shot QWK should be low -- that is the point: it demonstrates the CFP models' transfer is real and
modality-matched, not an artefact of the metric. Its few-shot row is a bonus probe of whether UWF-learned
retinal features are still linearly useful for CFP grading.

### Reproduce / scale up
- `QUICK_PASS=False` -> full k-grid `[1,5,10,20]`, 5 episodes, whole test sets.
- `FINETUNE=True` -> (todo hook) full fine-tune few-shot variant.
- All numbers land in `results/zeroshot.json`, `results/fewshot.json`, and `plots/`.

### Required attached inputs (Kaggle "Add Data")
1. `aptos2019-blindness-detection` (competition data).
2. An IDRiD "Disease Grading" dataset (any mirror with the grading labels CSV + images).
3. `krrish1008/checkpoint-retfound-512` (RETFound-CFP fine-tuned weights).
4. A small dataset holding the two local checkpoints: `best_multiscale_resnet.pth` and
   `best_model (for 3).pth` (upload from the repo's `findings/` and `contrastive_learning_lesions_3/`).